# TinyML Vehicle Classification with TensorFlow

Classes: Car, Motorcycle, Truck, Bus

This notebook covers the basic workflow: dataset → preprocessing → CNN → training → validation → evaluation → TFLite conversion.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

## 1. Dataset paths

Organize your dataset like this:

```text
vehicle_dataset/
├── train/
│   ├── car/
│   ├── motorcycle/
│   ├── truck/
│   └── bus/
└── validation/
    ├── car/
    ├── motorcycle/
    ├── truck/
    └── bus/
```

In [ ]:
train_dir = "vehicle_dataset/train"
validation_dir = "vehicle_dataset/validation"

IMG_SIZE = (96, 96)
BATCH_SIZE = 32

## 2. Load the datasets

In [ ]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    validation_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_dataset.class_names
print(class_names)

## 3. Build a small CNN

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255, input_shape=(96, 96, 3)),

    tf.keras.layers.Conv2D(16, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(4, activation="softmax")
])

model.summary()

## 4. Compile the model

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

## 5. Train

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=10
)

## 6. Plot training history

In [ ]:
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

## 7. Convert the trained model to TensorFlow Lite for TinyML

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("vehicle_classifier.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model saved!")

## 8. Next TinyML step

For ESP32-CAM deployment, we will later add **INT8 quantization**, convert the `.tflite` model into a C/C++ byte array, and run inference on the ESP32-CAM.